<a href="https://colab.research.google.com/github/Kentakure/hazardmap/blob/main/Hazard_map_in_Japan_using_only_GSI_tiles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import folium
from folium import plugins
import geopandas as gpd
import pandas as pd
from datetime import datetime
from google.colab import files

#地図の中心を設定
#https://geoshape.ex.nii.ac.jp/ka/resource/25/25202140002.html
center=[35.215976,136.12398]

# OpenStreetMap
fmap1 = folium.Map(location=center,
                   zoom_start = 10,
                   zoom_control=False, # ズームボタンを非表示
                   scrollWheelZoom=False, # マウスホイールでのズームを無効化
                   doubleClickZoom=False, # ダブルクリックでのズームを無効化
                   #dragging=False # ドラッグでの移動を無効化
)

# 災害伝承・避難場所等
# 出典：国土地理院 地理院地図 地理院タイル一覧
# https://maps.gsi.go.jp/development/ichiran.html
skhb_layers = {
    'shinsaidenshoushisetsu': '震災伝承施設', # 震災伝承施設
    'skhb01': '指定緊急避難場所（洪水）', # 指定緊急避難場所
    'skhb02': '指定緊急避難場所（崖崩れ、土石流及び地滑り）',
    'skhb03': '指定緊急避難場所（高潮）',
    'skhb04': '指定緊急避難場所（地震）',
    'skhb05': '指定緊急避難場所（津波）',
    'skhb06': '指定緊急避難場所（大規模な火事）',
    'skhb07': '指定緊急避難場所（内水氾濫）',
    'skhb08': '指定緊急避難場所（火山現象）',
    'sih': '指定避難所（一般）', # 指定避難所
    'sfh': '指定避難所（福祉）',
    'vdpf_fuji': '火山防災関連施設', # 火山防災関連施設
    'disaster_lore_all': '自然災害伝承碑' # 自然災害伝承碑
}
for url_part, name in skhb_layers.items():
    folium.raster_layers.TileLayer(
        tiles=f'https://cyberjapandata.gsi.go.jp/xyz/{url_part}/{{z}}/{{x}}/{{y}}.geojson',
        fmt='image/png',
        attr='&copy; <a href="https://maps.gsi.go.jp/development/ichiran.html" target="_blank" rel="noopener">国土地理院</a> ',
        name = name,
        tms=False,
        overlay=True,
        control=True,
        opacity=0.7
    ).add_to(fmap1)

# Base Maps
base_maps = [
    ('https://cyberjapandata.gsi.go.jp/xyz/std/{z}/{x}/{y}.png', '地理院地図', '&copy; <a href="https://maps.gsi.go.jp/development/ichiran.html" target="_blank" rel="noopener">国土地理院</a> '),
    ('https://cyberjapandata.gsi.go.jp/xyz/english/{z}/{x}/{y}.png', 'English', '&copy; <a href="https://maps.gsi.go.jp/development/ichiran.html" target="_blank" rel="noopener">国土地理院</a> '),
    ('http://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', 'Esri Satellite', 'Esri'),
    ('https://cyberjapandata.gsi.go.jp/xyz/seamlessphoto/{z}/{x}/{y}.jpg', '全国最新写真（シームレス）', '&copy; <a href="https://maps.gsi.go.jp/development/ichiran.html" target="_blank" rel="noopener">国土地理院</a> ')
]
for tiles, name, attr in base_maps:
    folium.raster_layers.TileLayer(
        tiles=tiles,
        fmt='image/png' if 'gsi' in tiles else None, # GSI maps are PNG, Esri is not explicit but usually handles it
        attr=attr,
        name=name,
        overlay=False,
        control=True
    ).add_to(fmap1)

# Layer of Disasters
# 出典：ハザードマップポータルサイト
# https://disaportal.gsi.go.jp/hazardmap/copyright/opendata.html
disaster_layers = {
    '01_flood_l2_shinsuishin_kuni_data': '洪水浸水想定区域（想定最大規模）',
    '01_flood_l2_keizoku_data': '浸水継続時間（想定最大規模）',
    '01_flood_l2_kaokutoukai_hanran_data': '家屋倒壊等氾濫想定区域（氾濫流）',
    '01_flood_l2_kaokutoukai_kagan_data': '家屋倒壊等氾濫想定区域（河岸侵食）',
    '02_naisui_data': '内水（雨水出水）浸水想定区域',
    '03_hightide_l2_shinsuishin_data': '高潮浸水想定区域',
    '04_tsunami_newlegend_data': '津波浸水想定',
    '05_dosekiryukeikaikuiki': '土砂災害警戒区域（土石流）',
    '05_kyukeishakeikaikuiki': '土砂災害警戒区域（急傾斜地の崩壊）',
    '05_jisuberikeikaikuiki': '土砂災害警戒区域（地すべり）',
    '05_nadarekikenkasyo': '雪崩危険箇所'
}
for url_part, name in disaster_layers.items():
    folium.raster_layers.TileLayer(
        tiles=f'https://disaportaldata.gsi.go.jp/raster/{url_part}/{{z}}/{{x}}/{{y}}.png',
        fmt='image/png',
        attr='&copy; <a href="https://disaportal.gsi.go.jp/hazardmap/copyright/opendata.html" target="_blank" rel="noopener">ハザードマップポータルサイト</a> ',
        name = name,
        tms=False,
        overlay=True,
        control=True,
        opacity=0.7
    ).add_to(fmap1)

#Layer Control
folium.LayerControl().add_to(fmap1)

#Fullscreen
folium.plugins.Fullscreen(
    position="topright",
    title="Expand me",
    title_cancel="Exit me",
    force_separate_button=True,
    ).add_to(fmap1)

#出力された地図fmap1をHTMLファイルに保存する
filename = f"Hazard_map_in_Japan_using_only_GSI_tiles_{datetime.now().strftime("%Y-%m-%d_%H-%M")}.html"
fmap1.save(filename)
files.download(filename)

#完成した地図fmap1を表示させる
fmap1

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>